In [4]:
from pathlib import Path
import sys
import os

%load_ext autoreload
%autoreload 2

dir = Path().resolve().parents[1]

if dir not in sys.path:
    print("directory path is not in the system path")
    sys.path.append(str(dir))
    print("adding directory...")
else:
    print("Directory already exists in the system path")

Failed to read module file 'C:\Users\User\anaconda3\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\CodingHenry\research_MBKM\venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\CodingHenry\research_MBKM\venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1140, in _find_and_load_unlocked
ModuleNotFoundE

directory path is not in the system path
adding directory...


In [5]:
from nn import Unet1D, Returns, RMSELoss
from scripts import train
from utils import log_transform
import torch
from torch.utils.data import DataLoader, random_split
import yfinance as yf
import pandas as pd
import numpy as np
import math

torch.Size([32, 4])
torch.Size([32, 4, 1])


In [6]:
ticker = "^GSPC"
start_interval = "2016-12-01"
end_interval = "2026-01-01"
interval = "1d"

raw_snp500 = torch.tensor(yf.Ticker(ticker).history(start=start_interval, end=end_interval, interval=interval)["Close"].to_numpy())

In [7]:
split_idx = len(raw_snp500) - math.ceil(len(raw_snp500) * 0.2)
train_raw_snp500, test_raw_snp500 = raw_snp500[:split_idx], raw_snp500[split_idx:]

window_size = 32

train_data = Returns(
  raw_returns=train_raw_snp500,
  window_size=window_size,
  transform=log_transform
)
test_data = Returns(
  raw_returns=test_raw_snp500,
  window_size=window_size,
  transform=log_transform
)

len(train_data), len(test_data)

(1794, 425)

In [8]:
train_dataloader = DataLoader(train_data, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=True)

next(iter(train_dataloader)).dtype

torch.float32

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
torch.cuda.manual_seed(42)
device

'cpu'

In [10]:
betas = torch.linspace(1e-4, 2e-2, 1000)
type(1 - betas)

torch.Tensor

In [107]:
encoder_in_channels = [1, 4, 8, 16]
encoder_out_channels = [4, 8, 16, 32]
decoder_in_channels = [32, 16, 8, 4]
decoder_out_channels = [16, 8, 4, 1]
attn_res = 16
n_res_block = 2
T = 1000
num_heads = 4
betas = torch.linspace(1e-4, 2e-2, T)
alpha_hats = torch.cumprod(
  input=1-betas,
  dim=0,
  dtype=torch.float32
)

model = Unet1D(
  attn_res=attn_res,
  n_res_block=n_res_block,
  encoder_in_channels=encoder_in_channels,
  encoder_out_channels=encoder_out_channels,
  decoder_in_channels=decoder_in_channels,
  decoder_out_channels=decoder_out_channels,
  T=T,
  num_heads=num_heads
)

In [108]:
loss_fn = RMSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [110]:
print(device)
train(
  train_data=train_dataloader,
  test_data=test_dataloader,
  optimizer=optimizer,
  loss_fn=loss_fn,
  epochs=10,
  alpha_hats=alpha_hats,
  model=model,
  scheduler=None,
  early_stopping=None,
  device=device,
  T=T
)

cpu


  0%|          | 0/10 [00:00<?, ?it/s]

passed attention
passed encoder x :  torch.Size([32, 32, 2])
passed attention
passed bottleneck x :  torch.Size([32, 32, 2])
skip u-net :  torch.Size([32, 32, 4])
x before concat :  torch.Size([32, 32, 4])
x after concat :  torch.Size([32, 64, 4])


RuntimeError: Given groups=1, weight of size [16, 32, 3], expected input[32, 64, 4] to have 32 channels, but got 64 channels instead